# AB-EMCTS on the R7615 — run everything from here

Companion to `analysis/AB_EMCTS_R7615_PLAN.md` (what each stage measures and why) and
`analysis/AB_EMCTS_RUNBOOK.md` (what to check before continuing). This notebook is the thing that
runs them.

**Run from `Tribes-master/`.** Every cell is independently re-runnable: the runners resume from their
checkpoints, so re-executing a stage continues it rather than restarting it.

**Order matters.** Stage A gates everything. Stage B comes before D — it pins the selection substrate
that D, E, F and G are all measured on.

## Parameters

Set these once. `WORKERS` comes from Stage A0's thread calibration; until then the ceiling
`ServerProfile` reports is a reasonable placeholder. `MACHINE_FACTOR` scales the per-turn runtime
model in the run banners — 1.0 is the dev laptop, the R7615 wants roughly 1.6, and A0 measures it.

In [ ]:
import os, sys, subprocess, pathlib, datetime

BASE           = "test_results/stage1"      # isolation-phase output root (the portfolio lives here)
STAGE0_BASE    = "test_results/stage0"      # Stage C's logged games
RESULTS        = "analysis/ab_results"      # analysis output
WORKERS        = 48                         # <- from Stage A0. Placeholder until then.
HEAP           = "64g"                      # -Xmx for run JVMs
CALIB_HEAP     = "32g"                      # -Xmx for the calibration JVM
MACHINE_FACTOR = "1.6"                      # <- from Stage A0/B measured PlanMillis
LEVELS         = "40000,80000,400000"
CHAMPION       = "Short5_e25"               # <- Stage D's winner, before running E and F

CP   = os.pathsep.join(["out", "lib/*"])    # ':' on Linux, ';' on Windows — os.pathsep gets it right
PY   = sys.executable                       # the interpreter running this notebook
LOGS = pathlib.Path("logs"); LOGS.mkdir(exist_ok=True)


def run(cmd, log=None, check=False):
    """Run a command, stream it here, and tee it to logs/<log>.log.

    Teeing matters: these stages run for hours, and a dropped SSH connection or a closed browser
    would otherwise take the only copy of the output with it.
    """
    path = LOGS / f"{log or 'cell'}.log"
    print(f"$ {cmd}\n  -> {path}\n", flush=True)
    started = datetime.datetime.now()
    with open(path, "w", encoding="utf-8") as fh:
        p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                             stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in p.stdout:
            print(line, end="")
            fh.write(line)
        rc = p.wait()
    print(f"\n[exit {rc}] after {datetime.datetime.now() - started}", flush=True)
    if check and rc != 0:
        raise RuntimeError(f"{cmd} exited {rc} — see {path}")
    return rc


def java(main, args="", heap=None, props="", log=None, check=False):
    return run(f'java -Xmx{heap or HEAP} {props} -cp "{CP}" {main} {args}',
               log=log or main.split(".")[-1], check=check)


print("cwd:", os.getcwd())
assert pathlib.Path("lib").is_dir(), "run this notebook from Tribes-master/"

---
## 0. Build, and the shakedown

`--release 11` pins the language level *and* the API set to the server's JDK, so anything newer is a
compile error here rather than a surprise on the server.

In [ ]:
run("find src -name '*.java' > sources.txt")
run('javac --release 11 -nowarn -d out -cp "lib/*" @sources.txt && echo BUILD OK',
    log="build", check=True)
run("echo classes: $(find out -name '*.class' | wc -l)")

### The shakedown — ~6 min, 18 checks

Stage A is a ~5.6 h block that is mostly single-threaded and needs an idle machine, so an environment
problem found inside it costs an evening. This exercises every moving part at toy size. It measures
nothing.

**Read two things by eye even on a pass:** cores must be **detected**, not `ASSUMED`; and
`test_results/` must be on local disk, because the calibration is a timing measurement.

In [ ]:
run("./run_ab_shakedown.sh", log="shakedown")

---
## Stage A — machine, currency, gates  *(~5.6 h; nothing measured)*

### A0 — what this machine will do

In [ ]:
java("core.data.exp.ServerProfile", heap="4g", log="A0_profile")

### A1 — the portfolio and the frozen split *(~0.3 h)*

400 scenarios, built **once** and pinned for the whole line: `visitSeed` keys on scenario id, so a
rebuild silently re-pairs everything.

Read the freeze report before trusting the split — the halves should differ by at most one scenario
and a couple of branching points, not by twenty.

In [ ]:
java("core.data.exp.BuildV6Portfolio", f"{BASE}/_portfolio {WORKERS} 200 200",
     log="A1_portfolio", check=True)
java("core.data.exp.PortfolioSplit", f"{BASE}/_portfolio", heap="4g", log="A1_split", check=True)

### A0b — thread calibration, per budget level

Tree size grows with the budget, so the best thread count at 400k is not the one at 40k. Record `W`
per level, and read `ThrottlePct` / `ClockDropPct`: a point that only reaches peak on borrowed boost
headroom is not the one to run for hours.

**Set `WORKERS` in the parameters cell from this, then re-run that cell.**

In [ ]:
java("core.data.exp.ThreadCalibrationV6",
     f"{BASE}/_portfolio {BASE}/_calibration {LEVELS}", log="A0_threads")

### A2 — calibrate idle *(~2.6 h, single-threaded)*

54 cells: the reference `EMCTS` plus all 17 action-bias arms, x 3 levels.

**Nothing else may run on this machine while this does.** `auto` keeps the suite's own turn counts
(`8,4,2`) — do not replace it with a number unless you mean to, because the zero-eval arms make ~4x
fewer copies by construction and need the extra turns at 40k to clear the sample floor. Verdict must
read `PASS`; on `FAIL` the weights are deliberately not written and the measurements are still in
`compute_profiles.csv`.

In [ ]:
java("core.data.exp.ComputeCalibrateV6",
     f"{BASE}/_portfolio {BASE}/_calibration 3 auto 6 {LEVELS} --suite ab",
     heap=CALIB_HEAP, log="A2_calibrate")

### A3' — calibrate under load *(~1.0 h)*

Does a per-configuration cost ratio survive the move from an idle machine to a loaded one? The arms
differ several-fold in allocation rate, and GC is nearly free on one thread while it competes for
cores, L3 and memory bandwidth at `W`.

Writes `compute_weights_load<W>.properties` **beside** the idle weights, never over them. **Keep the
verdict block** — it decides whether the wall-clock fairness re-test is a secondary check or
co-primary for the whole line.

In [ ]:
java("core.data.exp.ComputeCalibrateV6",
     f"{BASE}/_portfolio {BASE}/_calibration 3 auto 6 {LEVELS} --suite a3 --underLoad {WORKERS}",
     heap=CALIB_HEAP, log="A3_loaded")

### A4 — the gates

Three questions: is Baseline still bit-identical to plain EMCTS, is the Stage-0 instrumentation still
inert, and **does the bias actually have evidence to act on at each budget**.

That last one is the floor probe. The action-bias line fails quietly from *below*: `qhat` shrinks
toward the running global mean, so when every candidate key is unseen they all tie and the tie-break
is uniform — the arm becomes Baseline with extra bookkeeping and still reports a small noisy effect
rather than announcing that the mechanism never fired. It is weight-dependent, which is why it runs
**after** A2.

In [ ]:
java("core.data.exp.Stage1Runner", "verify", heap="8g", log="A4_verify_stage1")
java("core.data.exp.Stage0SignalRunner", "verify 40000 20", heap="8g", log="A4_verify_stage0")
java("core.data.exp.ActionBiasFloorProbe",
     f"{BASE}/_portfolio {BASE}/_calibration {LEVELS} 20 {WORKERS}", log="A4_floor")

> ### Gate A
> Both verifies PASS &middot; A2 verdict PASS &middot; A3' verdict recorded &middot; floor probe
> passes at 80k &middot; split frozen and balanced &middot; cores detected.
> **Nothing downstream runs otherwise.**

A level that fails only the floor probe is a *reporting* level — it can carry a budget trend or an
anytime curve, but not a claim that the bias helped, because at that budget the bias was not
operating.

---
## Stage B — the substrate  *(~0.5 h)* — run this before D

Four arms at 80k. The two plus arms are the real `Baseline` and `Bias_e25` rather than renamed
copies, so Stage D later **skips them on resume** — same checkpoint keys, no work done twice.

In [ ]:
java("core.data.exp.Stage1Runner", f"{WORKERS} {BASE} dev B",
     props=f"-Dstage1.machineFactor={MACHINE_FACTOR}", log="B")
run(f'{PY} analysis/analyze_stage1.py {BASE}/dev --outdir {RESULTS}/B', log="B_analysis")

In [ ]:
print(open(f"{RESULTS}/B/SUMMARY.md", encoding="utf-8").read())

> ### Gate B — read two things, and stop until you have them
> 1. **The main effect** `Comma_Off - Plus_Off`. Whichever wins becomes the substrate *and the
>    baseline* for D-G. If comma wins, that is a re-pin, not a footnote.
> 2. **The design section** — the measured ICC, which sets Stage D's visit count. The default is 8.

---
## Stage C — is there still a signal?  *(~0.7 h, observation only)*

In [ ]:
java("core.data.exp.Stage0SignalRunner",
     f"{WORKERS} {LEVELS} 48 emcts 50 {STAGE0_BASE}", log="C")
run(f'{PY} analysis/analyze_stage0.py {STAGE0_BASE} --outdir {RESULTS}/C', log="C_analysis")

In [ ]:
print(open(f"{RESULTS}/C/SUMMARY.md", encoding="utf-8").read())

> ### Gate C
> Mean rho >= 0.15 with a CI excluding zero, at 80k, for at least one granularity. **Fails everywhere
> -> stop the line**; the premise is gone and D-G are unfundable.

Also read R9 and R10 — payoff versus expansion-target age, and the re-measured duplicate rate.

---
## Stage D — biased insertion  *(~5.3 h)*

All 11 arms at 80k -> 40k -> 400k. That order is the schedule, not a listing: leaves are checkpointed
independently, so an interrupted run keeps the main grid and its budget trend rather than having
spent the time on the expensive confirmation first.

In [ ]:
java("core.data.exp.Stage1Runner", f"{WORKERS} {BASE} dev D",
     props=f"-Dstage1.machineFactor={MACHINE_FACTOR}", log="D")
run(f'{PY} analysis/analyze_stage1.py {BASE}/dev --outdir {RESULTS}/D', log="D_analysis")

In [ ]:
print(open(f"{RESULTS}/D/SUMMARY.md", encoding="utf-8").read())

> ### Gate D
> The champion is the best arm whose CI excludes 0 **and** which survives the wall-clock re-test.
> **Write epsilon\* and the champion down before Stage F**, and set `CHAMPION` in the parameters
> cell. If epsilon\* is materially different from 0.25, change `Stage1Runner.Z0_EPS` **and re-run the
> `ab` calibration suite** — the arms and their prices have to move together.

---
## Stage E — table-directed repair  *(~2.6 h)*

Check `RepairCandidateEvals` reads **0** for every `Z0_*` arm. That is a construction check, not a
finding: a non-zero value means the mode is not engaged and the run is a bug.

In [ ]:
java("core.data.exp.Stage1Runner", f"{WORKERS} {BASE} dev E {CHAMPION}",
     props=f"-Dstage1.machineFactor={MACHINE_FACTOR}", log="E")
run(f'{PY} analysis/analyze_stage1.py {BASE}/dev --outdir {RESULTS}/E', log="E_analysis")

---
## Stage F — the holdout  *(~2.2 h, runs once)*

**Freeze the choices in writing first.** The holdout is worth nothing if it is read twice.

In [ ]:
java("core.data.exp.Stage1Runner", f"{WORKERS} {BASE} holdout F {CHAMPION}",
     props=f"-Dstage1.machineFactor={MACHINE_FACTOR}", log="F")
run(f'{PY} analysis/analyze_stage1.py {BASE}/holdout --outdir {RESULTS}/F', log="F_analysis")

In [ ]:
print(open(f"{RESULTS}/F/SUMMARY.md", encoding="utf-8").read())

> ### Gate F
> A development effect that does not replicate here **is not a result**. Report both numbers side by
> side, always.

---
## Stage G — the yardstick

Needs work items 9 and 10 (map set 25 -> 75, and the matchup list plus the errors-in-variables
mapping), which are not written yet and depend on Stage D's champion. Size it from a 4-game timing
probe rather than from the plan: full-game turns are far cheaper than isolation turns, and this
line's own bug table records a runtime estimate wrong by 10x in exactly that direction.

---
## What to keep

Small, and it is what makes a result reproducible rather than merely reported.

In [ ]:
run("git rev-parse HEAD", log="provenance")
for p in [f"{BASE}/_calibration/compute_profiles.csv",
          f"{BASE}/_calibration/compute_weights.properties",
          f"{BASE}/_calibration/thread_calibration.csv",
          f"{BASE}/_calibration/action_bias_floor.csv",
          f"{BASE}/_portfolio/portfolio.csv",
          f"{BASE}/_portfolio/_split_dev.txt",
          f"{BASE}/_portfolio/_split_holdout.txt"]:
    print(("  ok  " if pathlib.Path(p).exists() else "  --  ") + p)
print("\nlogs/ holds every run's console output; " + RESULTS + "/ holds the tables and summaries.")